<a href="https://colab.research.google.com/github/matheusmanfron/Projeto-I.A-2.0/blob/main/ModeloKNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

janela_curta = 5     # últimos 5 jogos
janela_longa = 10    # últimos 10 jogos
min_jogados = 3        # mínimo de jogos anteriores para calcular a média
proporcao_treino = 0.80
nome_classes = {0: "derrota", 1: "empate", 2: "vitória"}

df = pd.read_csv("Partidas_Tratadas.csv", parse_dates=["data"])
df = df.sort_values(["data", "fixture_id", "team_id"]).reset_index(drop=True)
df["pontos"] = df["resultado"].map({0: 0, 1: 1, 2: 3})
df["mando"] = (df["local_visitante"] == "home").astype(int)
df.shape

(3446, 19)

In [4]:
# --- Forma em pontos para TODOS os times (inclusive os que só aparecem como adversário) ---
a = df[["fixture_id", "data", "team_id", "pontos"]]
b = df[["fixture_id", "data", "adversario_id", "resultado"]].rename(columns={"adversario_id": "team_id"})
b["pontos"] = b["resultado"].map({0: 3, 1: 1, 2: 0})   # resultado visto pelo adversário
longa = (pd.concat([a, b[["fixture_id", "data", "team_id", "pontos"]]])
           .drop_duplicates(["fixture_id", "team_id"])
           .sort_values(["team_id", "data", "fixture_id"]))

def media_movel(s, janela):
    return s.shift(1).rolling(janela, min_periods=min_jogados).mean()   # shift(1): exclui o jogo atual

g = longa.groupby("team_id")["pontos"]
longa["pts_5"] = g.transform(lambda s: media_movel(s, janela_curta))
longa["pts_10"] = g.transform(lambda s: media_movel(s, janela_longa))
forma_pts = longa[["fixture_id", "team_id", "pts_5", "pts_10"]]

# --- Estatísticas do próprio time (só times com linhas próprias na base) ---
ordenado = df.sort_values(["team_id", "data", "fixture_id"])
estatisticas = {"Gols": "gols_5", "Escanteios": "escanteios_5", "Posse de bola (%)": "posse_5",
                "Cartões amarelos": "amarelos_5", "Cartões vermelhos": "vermelhos_5"}
for original, nova in estatisticas.items():
    df[nova] = ordenado.groupby("team_id")[original].transform(lambda s: media_movel(s, janela_curta))

# --- Junta forma do time e do adversário ---
df = df.merge(forma_pts, on=["fixture_id", "team_id"], how="left")
df = df.merge(forma_pts.rename(columns={"team_id": "adversario_id", "pts_5": "adv_pts_5", "pts_10": "adv_pts_10"}),
              on=["fixture_id", "adversario_id"], how="left")
df["dif_pts_10"] = df["pts_10"] - df["adv_pts_10"]

FEATURES = ["mando", "pts_5", "pts_10", "gols_5", "escanteios_5", "posse_5",
            "amarelos_5", "vermelhos_5", "adv_pts_5", "adv_pts_10", "dif_pts_10"]

antes = len(df)
df = df.dropna(subset=FEATURES).reset_index(drop=True)
print(f"{antes - len(df)} linhas removidas por falta de histórico; restam {len(df)}")
df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(2)

82 linhas removidas por falta de histórico; restam 3274


,mean,std,min,max
mando,0.50,0.50,0.0,1.00
pts_5,1.47,0.72,0.0,3.00
pts_10,1.47,0.61,0.0,3.00
gols_5,1.42,0.72,0.0,4.60
escanteios_5,5.42,1.75,1.4,15.00
posse_5,51.17,8.94,27.2,78.80
amarelos_5,1.84,0.66,0.0,4.25
vermelhos_5,0.10,0.14,0.0,0.80
adv_pts_5,1.37,0.73,0.0,3.00
adv_pts_10,1.37,0.62,0.1,3.00


In [6]:
#Divisão do modelo, 80% de treino são das partidas mais antigas, e 20% de teste das mais recentes

df = df.sort_values(["data", "fixture_id"]).reset_index(drop=True)
data_corte = df["data"].iloc[int(len(df) * proporcao_treino)]
treino = df[df["data"] <= data_corte]
teste = df[df["data"] > data_corte]

X_treino, y_treino = treino[FEATURES], treino["resultado"]
X_teste, y_teste = teste[FEATURES], teste["resultado"]
print(f"Treino: {len(treino)} linhas (até {data_corte.date()})")
print(f"Teste:  {len(teste)} linhas ({teste['data'].min().date()} a {teste['data'].max().date()})")
print("\nProporção das classes no treino:")
print(y_treino.map(nome_classes).value_counts(normalize=True).round(3))

Treino: 2620 linhas (até 2024-12-07)
Teste:  654 linhas (2024-12-08 a 2026-05-17)

Proporção das classes no treino:
resultado
vitória    0.416
derrota    0.351
empate     0.233
Name: proportion, dtype: float64


In [8]:
pipeline = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier())])

grade = {
    "knn__n_neighbors": [5, 15, 25, 35, 51, 75, 101, 151],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],        # 1 = Manhattan, 2 = Euclidiana
}

busca = GridSearchCV(pipeline, grade, cv=TimeSeriesSplit(n_splits=5),
                     scoring="accuracy", n_jobs=-1)
busca.fit(X_treino, y_treino)

print("Melhores parâmetros:", busca.best_params_)
print(f"Acurácia na validação cruzada: {busca.best_score_:.3f}")
(pd.DataFrame(busca.cv_results_)
   [["param_knn__n_neighbors", "param_knn__weights", "param_knn__p", "mean_test_score"]]
   .sort_values("mean_test_score", ascending=False).head(8).round(3))

Melhores parâmetros: {'knn__n_neighbors': 75, 'knn__p': 2, 'knn__weights': 'uniform'}
Acurácia na validação cruzada: 0.532


,param_knn__n_neighbors,param_knn__weights,param_knn__p,mean_test_score
22,75,uniform,2,0.532
24,101,uniform,1,0.531
26,101,uniform,2,0.530
25,101,distance,1,0.529
27,101,distance,2,0.529
20,75,uniform,1,0.528
23,75,distance,2,0.528
15,35,distance,2,0.528


In [9]:
modelo = busca.best_estimator_
pred = modelo.predict(X_teste)

# Baseline 1: sempre a classe mais frequente. Baseline 2: mandante vence, visitante perde.
base1 = DummyClassifier(strategy="most_frequent").fit(X_treino, y_treino).predict(X_teste)
base2 = np.where(X_teste["mando"] == 1, 2, 0)

resultados = pd.DataFrame({
    "acurácia": [accuracy_score(y_teste, p) for p in (base1, base2, pred)],
    "F1 macro": [f1_score(y_teste, p, average="macro") for p in (base1, base2, pred)],
}, index=["Baseline: classe mais frequente", "Baseline: mando", "KNN"]).round(3)
resultados

,acurácia,F1 macro
Baseline: classe mais frequente,0.405,0.192
Baseline: mando,0.469,0.354
KNN,0.508,0.392


In [13]:
print(classification_report(y_teste, pred, target_names=list(nome_classes.values()), zero_division=0))
pd.DataFrame(confusion_matrix(y_teste, pred),
             index=[f"real: {v}" for v in nome_classes.values()],
             columns=[f"previsto: {v}" for v in nome_classes.values()])

              precision    recall  f1-score   support

     derrota       0.50      0.67      0.57       238
      empate       0.18      0.01      0.02       151
     vitória       0.52      0.65      0.58       265

    accuracy                           0.51       654
   macro avg       0.40      0.44      0.39       654
weighted avg       0.44      0.51      0.45       654



,previsto: derrota,previsto: empate,previsto: vitória
real: derrota,159,4,75
real: empate,68,2,81
real: vitória,89,5,171


In [14]:
modelo_final = clone(modelo).fit(df[FEATURES], df["resultado"])

# Forma atual de cada time (últimos jogos da base inteira)
pts_atual = longa.groupby("team_id")["pontos"].agg(
    pts_5=lambda s: s.tail(janela_curta).mean(), pts_10=lambda s: s.tail(janela_longa).mean())
est_atual = pd.DataFrame({nova: ordenado.groupby("team_id")[orig].apply(lambda s: s.tail(janela_curta).mean())
                          for orig, nova in estatisticas.items()})

ids = {**dict(zip(df["time"], df["team_id"])), **dict(zip(df["adversario"], df["adversario_id"]))}
times_com_estatisticas = set(df["time"])

def prever(time, adversario, mando):
    """mando = 'home' se `time` joga em casa, 'away' se joga fora."""
    if time not in times_com_estatisticas:
        raise ValueError(f"{time} não tem estatísticas próprias na base; escolha entre {sorted(times_com_estatisticas)}")
    t, a = ids[time], ids[adversario]
    x = {"mando": int(mando == "home"),
         "pts_5": pts_atual.loc[t, "pts_5"], "pts_10": pts_atual.loc[t, "pts_10"],
         **est_atual.loc[t].to_dict(),
         "adv_pts_5": pts_atual.loc[a, "pts_5"], "adv_pts_10": pts_atual.loc[a, "pts_10"]}
    x["dif_pts_10"] = x["pts_10"] - x["adv_pts_10"]
    proba = modelo_final.predict_proba(pd.DataFrame([x])[FEATURES])[0]
    return pd.Series(proba, index=[nome_classes[c] for c in modelo_final.classes_]).round(3)

print("Última data na base:", df["data"].max().date())
print("\nCeltic (casa) x Rangers:");  print(prever("Celtic", "Rangers", "home"))
print("\nAberdeen (fora) x Hearts:"); print(prever("Aberdeen", "Hearts", "away"))

Última data na base: 2026-05-17

Celtic (casa) x Rangers:
derrota    0.160
empate     0.187
vitória    0.653
dtype: float64

Aberdeen (fora) x Hearts:
derrota    0.587
empate     0.187
vitória    0.227
dtype: float64
